In [ ]:
from langchain_core.documents import Document
from typing import List, Dict
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

/tmp/ipykernel_51410/1393003851.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
/home/jaywardhan/RAG_Udemy/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
class SmartPDFProcessor:

    def __init__(self, chunk_size = 1000, chunk_overlap = 100):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.text_splitter = RecursiveCharacterTextSplitter(
            separators = [" "],
            chunk_size = chunk_size,
            chunk_overlap = chunk_overlap,
            length_function = len
        )

    def pdf_processor(self, path: str) -> List[Document]:

        pdf_loader = PyPDFLoader(path)

        pages= pdf_loader.load()

        processed_chunks = []

        for i , page in enumerate(pages):

            cleaned_txt = self._clean_text(page.page_content)

            if len(cleaned_txt.strip()) < 50:
                continue

            chunks = self.text_splitter.create_documents(
                texts = [cleaned_txt],
                metadatas = [{
                    **page.metadata,
                    'page_no' : i+1,
                    'pdf_processing_method' : "SmartPdfProcessor",
                    'Number_of_characters' : len(cleaned_txt),
                    'total_pages' : len(pages)

                }]
            )

            processed_chunks.extend(chunks)

        return processed_chunks

    def _clean_text(self, text: str) -> str:

        text = " ".join(text.split())

        text = text.replace("ﬁ","fi")
        text = text.replace("æ","Ae")
        text = text.replace("ﬂ","fl")
        text = text.replace("ﬃ","ffi")
        text = text.replace("ﬄ","ffl")
        text = text.replace("œ","oe")

        return text



processor = SmartPDFProcessor()

# try:

smart_chunks = processor.pdf_processor("data/pdfs/rag.pdf")

print(f"Total Number of chunks: {len(smart_chunks)}")

for i , chunk in enumerate(smart_chunks[:3]):
    print(f"chunk1: {i+1}")
    print(f"Metadata: {chunk.metadata}")
    print(f"Content: {chunk.page_content}\n")

# except Exception as e:
#     print(f"Processing Error: {e}")


            


    

Total Number of chunks: 87
chunk1: 1
Metadata: {'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-01-06T08:23:25-08:00', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': '2026-01-06T08:23:25-08:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'source': 'data/pdfs/rag.pdf', 'total_pages': 25, 'page': 0, 'page_label': '1', 'page_no': 1, 'pdf_processing_method': 'SmartPdfProcessor', 'Number_of_characters': 2856}
Content: Speech and Language Processing. Daniel Jurafsky & James H. Martin. Copyright © 2026. All rights reserved. Draft of January 6, 2026. CHAPTER 11 Information Retrieval and Retrieval-Augmented Generation On two occasions I have been asked,—“Pray, Mr. Babbage, if you put into the machine wrong figures, will the right answers come out?” ... I am not able rightly to apprehend the kind of confusion of ideas that could provoke such a question